# Price Optimization Analysis for Revenue Maximization

This notebook analyzes how to maximize revenue by adjusting product prices based on demand forecasting models.

In [2]:
pip install numpy pandas matplotlib seaborn calplot mplcyberpunk catboost scikit-learn holidays joblib

Looking in indexes: https://artifactory.tcsbank.ru/artifactory/api/pypi/python-all/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.3/132.3 kB 12.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 72.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 122.9 MB/s eta 0:00:0000:010:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 110.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.1/309.1 kB 121.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 31.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.7/37.7 MB 123.0 MB/s eta 0:00:0000:0100:01
  Created wheel for calplot: filename=calplot-0.1.7.5-py3-none-any.whl size=8218 sha256=789400fca2cea325cced30ef2754eccb472b939e8c5675e8cd1afc9e4e9ae58c
  Stored in directory: /home/mlcore/.cache/pip/wheels/3d/56/ba/39ec6023036b013e295d78390451fabc0238eaf6eee57fe7de
Successf

In [3]:
import numpy as np
import pandas as pd
import os
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import matplotlib.pyplot as plt
import seaborn as sns
from calplot import calplot as clp
import mplcyberpunk
plt.style.use("cyberpunk")

from catboost import CatBoostRegressor

from sklearn.metrics import root_mean_squared_error as RMSE
from sklearn.model_selection import train_test_split
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

import gc
import holidays

pd.set_option('display.float_format', lambda x: '%.4f' % x)

In [4]:
# Set up the project directory
# Use the notebook's directory as the base path
import os

# Get the directory where this notebook is located
# Files will be loaded relative to the notebook location
print("Current folder:", os.getcwd())
print("Directory contents:", os.listdir())

Current folder: /workdir
Directory contents: ['mlc_notebook_readme.md', '.ipynb_checkpoints', 'sber_h1.xlsx', 'eurusd_h1.xlsx', 'gold_h1.xlsx', 'catboost_retail_model.cbm', 'ML_TrendModelGPT_clean.ipynb', 'silver_h1.xlsx', '.virtual_documents', 'catboost_clf_1.cbm', 'mag', 'project_another', 'stores.csv', 'markdowns.csv', 'zoomcamp.ipynb', 'online.csv', 'sample_submission.csv', 'test.csv', 'price_history.csv', 'sales.csv', 'brent_h1.xlsx', 'discounts_history.csv', 'submission_baseline.csv', 'catboost_info', 'submission_catboost_best_clip.csv', 'top4-retail-demand-forecast-mlzc-compet-24.ipynb', 'catboost_clf_old.cbm', 'zoomcamp_modified_version.ipynb', 'submission.csv', 'translated_catalog.csv', 'catalog.csv', 'actual_matrix.csv', 'old_zoomcamp_modified_version_old.ipynb', 'revenue_optimization_implementation.ipynb', 'demand_forecasting_with_price_sensitivity.ipynb', 'price_optimization_analysis.ipynb', 'revenue_optimization_overview.ipynb']


In [5]:
# Load data files
files = {
    "sales": "sales.csv",
    "online": "online.csv",
    "markdowns": "markdowns.csv",
    "price_history": "price_history.csv",
    "discounts_history": "discounts_history.csv",
    "actual_matrix": "actual_matrix.csv",
    "catalog": "catalog.csv",
    "stores": "stores.csv",
    "test": "test.csv",
    "sample_submission": "sample_submission.csv",
}

def read_csv(name, **kwargs):
    fp = files[name]
    df = pd.read_csv(fp, **kwargs)
    return df

df_sales = read_csv("sales", index_col=0)
df_online = read_csv("online", index_col=0)
df_markdowns = read_csv("markdowns", index_col=0)
df_price_history = read_csv("price_history", index_col=0)
df_discounts_history = read_csv("discounts_history", index_col=0)
df_actual_matrix = read_csv("actual_matrix", index_col=0)
df_catalog = read_csv("catalog", index_col=0)
df_stores = read_csv("stores", index_col=0)

df_test = read_csv("test", sep=";", index_col="row_id")
df_test = df_test.reset_index()  # ADDED: row_id теперь обычная колонка и больше не потеряется
df_sample_submission = read_csv("sample_submission", index_col=0)

## Data Preparation for Price Analysis

In [6]:
# Optimize dtypes function
def optimizing_dtypes(df, nameCSV):
    bytesInOneMB = 1048576
    print(f"{nameCSV}: from {round(df.memory_usage().sum()/bytesInOneMB, 2)}MB", end="")

    col_float64 = df.select_dtypes(include=["float64"]).columns.values
    if len(col_float64) > 0:
        df[col_float64] = df[col_float64].astype("float32", copy=False)

    col_int64 = df.select_dtypes(include=["int64"]).columns.values
    if len(col_int64) > 0:
        for col in col_int64:
            uniq = df[col].dropna().unique()
            if len(uniq) <= 2 and set(uniq).issubset({0, 1}):
                df[col] = df[col].astype("int8", copy=False)
            else:
                df[col] = df[col].astype("int32", copy=False)

    print(f" reduced to {round(df.memory_usage().sum()/bytesInOneMB,2)}MB")
    return df

# Optimize dataframes
df_sales = optimizing_dtypes(df_sales, "sales.csv")
df_online = optimizing_dtypes(df_online, "online.csv")
df_markdowns = optimizing_dtypes(df_markdowns, "markdowns.csv")
df_price_history = optimizing_dtypes(df_price_history, "price_history.csv")
df_discounts_history = optimizing_dtypes(df_discounts_history, "discounts_history.csv")
df_actual_matrix = optimizing_dtypes(df_actual_matrix, "actual_matrix.csv")
df_catalog = optimizing_dtypes(df_catalog, "catalog.csv")
df_stores = optimizing_dtypes(df_stores, "stores.csv")
df_test = optimizing_dtypes(df_test, "test.csv")
df_sample_submission = optimizing_dtypes(df_sample_submission, "sample_submission.csv")

sales.csv: from 396.95MB reduced to 283.53MB
online.csv: from 60.0MB reduced to 42.85MB
markdowns.csv: from 0.48MB reduced to 0.34MB
price_history.csv: from 31.98MB reduced to 23.99MB
discounts_history.csv: from 257.27MB reduced to 185.81MB
actual_matrix.csv: from 1.07MB reduced to 0.94MB
catalog.csv: from 15.09MB reduced to 12.58MB
stores.csv: from 0.0MB reduced to 0.0MB
test.csv: from 26.97MB reduced to 20.23MB
sample_submission.csv: from 13.48MB reduced to 7.58MB


In [7]:
# Clean sales data
df_sales["date"] = pd.to_datetime(df_sales["date"])
df_sales["price_base"] = df_sales["sum_total"] / df_sales["quantity"]
df_sales = df_sales.sort_values(["date", "item_id", "store_id"])

mask = (df_sales["quantity"] <= 0) | (df_sales["price_base"] <= 0) | (df_sales["sum_total"] <= 0) | ~np.isfinite(df_sales["price_base"])
df_sales.drop(df_sales[mask].index, axis=0, inplace=True)

# Clean test data
df_test["date"] = pd.to_datetime(df_test["date"])

# Clean online data
df_online["date"] = pd.to_datetime(df_online["date"])
df_online["price_base"] = df_online["sum_total"] / df_online["quantity"]
df_online = df_online.sort_values(["date", "item_id", "store_id"])

mask = (df_online["quantity"] <= 0) | (df_online["price_base"] <= 0) | (df_online["sum_total"] <= 0) | ~np.isfinite(df_online["price_base"])
df_online.drop(df_online[mask].index, axis=0, inplace=True)

/tmp/ipykernel_332/2766753665.py:10: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df_test["date"] = pd.to_datetime(df_test["date"])


In [8]:
# Merge sales and online data
df_online = df_online.rename(columns={"price_base":"price_base_online", "sum_total":"sum_total_online"})
df_online["online"] = True

df = pd.merge(df_sales, df_online, on=["date", "item_id", "store_id"], how="outer", suffixes=("_x", "_y"))
df["quantity"] = df[["quantity_x", "quantity_y"]].sum(axis=1)

# Keep columns as in df_sales
df = df[df_sales.columns.to_list()]
df = df.fillna(0)

In [9]:
# Process discounts history for price elasticity analysis
df_discounts_history["date"] = pd.to_datetime(df_discounts_history["date"])

mask = (
    (df_discounts_history["sale_price_before_promo"] <= 0) |
    (df_discounts_history["sale_price_time_promo"] <= 0) |
    (df_discounts_history["date"] > df_test["date"].max())
)
df_discounts_history.drop(df_discounts_history[mask].index, axis=0, inplace=True)

df_discounts_history["promo_day"] = True
df_discounts_history["discount_percentage"] = (df_discounts_history["sale_price_before_promo"] - df_discounts_history["sale_price_time_promo"]) / df_discounts_history["sale_price_before_promo"] * 100

columns = df.columns.to_list()
df = pd.merge(
    df, df_discounts_history,
    how="left",
    left_on=["date", "item_id", "store_id"],
    right_on=["date", "item_id", "store_id"],
    suffixes=("_x", "_y")
)

df = df[columns + ["promo_day", "sale_price_before_promo", "sale_price_time_promo", "discount_percentage", "promo_type_code", "doc_id", "number_disc_day"]]
df["promo_day"] = df["promo_day"].fillna(False)

fill_cols = ["sale_price_before_promo", "sale_price_time_promo", "discount_percentage", "number_disc_day"]
df[fill_cols] = df[fill_cols].fillna(0)

## Price Elasticity Analysis

In [13]:
# Calculate price elasticity for items with discount history
def calculate_price_elasticity(df):
    # Предвычисляем статистики по нормальным дням (один раз)
    normal_stats = (df[df['promo_day'] == False]
                    .groupby(['item_id', 'store_id'])
                    .agg({
                        'quantity': 'mean',
                        'price_base': 'mean'
                    })
                    .rename(columns={
                        'quantity': 'avg_qty_normal', 
                        'price_base': 'avg_price_normal'
                    }))
    
    # Статистики по промо-дням
    promo_stats = (df[df['promo_day'] == True]
                   .groupby(['item_id', 'store_id'])
                   .agg({
                       'quantity': 'mean',
                       'sale_price_time_promo': 'mean'
                   })
                   .rename(columns={
                       'quantity': 'avg_qty_promo', 
                       'sale_price_time_promo': 'avg_price_promo'
                   }))
    
    # Объединяем только по группам с промо И нормальными днями
    merged = promo_stats.join(normal_stats, how='inner')
    
    # Вычисляем изменения и эластичность
    merged['qty_change_pct'] = ((merged['avg_qty_promo'] - merged['avg_qty_normal']) / 
                                merged['avg_qty_normal']) * 100
    merged['price_change_pct'] = ((merged['avg_price_promo'] - merged['avg_price_normal']) / 
                                  merged['avg_price_normal']) * 100
    merged['elasticity'] = merged['qty_change_pct'] / merged['price_change_pct']
    
    # Фильтруем валидные случаи
    result = merged[(merged['avg_qty_normal'] > 0) & 
                    (merged['avg_price_normal'] > 0) & 
                    (merged['price_change_pct'] != 0)].copy()
    
    return (result[['elasticity', 'avg_qty_normal', 'avg_qty_promo', 
                    'avg_price_normal', 'avg_price_promo']]
            .reset_index())


# Calculate elasticity
elasticity_df = calculate_price_elasticity(df)
print(f"Calculated elasticity for {len(elasticity_df)} item-store combinations")
elasticity_df.head()

Calculated elasticity for 30016 item-store combinations


,item_id,store_id,elasticity,avg_qty_normal,avg_qty_promo,avg_price_normal,avg_price_promo
0,00179dda14f8,1,-0.0000,1.0000,1.0000,662.4055,499.9000
1,001829cb707d,1,-6.6005,2.7182,5.8068,145.9415,120.8179
2,001829cb707d,2,-2.0932,1.6667,2.3361,156.0676,126.1185
3,001829cb707d,4,-5.9090,1.7966,4.0381,149.2148,117.7095
4,001b302a9387,1,-8.8473,5.2143,5.7600,78.5087,77.5800


In [14]:
# Categorize items by elasticity
def categorize_by_elasticity(elasticity_df):
    elasticity_df['elasticity_category'] = 'inelastic'
    elasticity_df.loc[elasticity_df['elasticity'] < -1, 'elasticity_category'] = 'elastic'
    elasticity_df.loc[(elasticity_df['elasticity'] >= -1) & (elasticity_df['elasticity'] < 0), 'elasticity_category'] = 'inelastic'
    elasticity_df.loc[elasticity_df['elasticity'] > 0, 'elasticity_category'] = 'abnormal'  # Quantity increases with price
    
    return elasticity_df

elasticity_df = categorize_by_elasticity(elasticity_df)
print("Elasticity categories distribution:")
print(elasticity_df['elasticity_category'].value_counts())
elasticity_df.head()

Elasticity categories distribution:
elasticity_category
elastic      15402
abnormal      8783
inelastic     5831
Name: count, dtype: int64


,item_id,store_id,elasticity,avg_qty_normal,avg_qty_promo,avg_price_normal,avg_price_promo,elasticity_category
0,00179dda14f8,1,-0.0000,1.0000,1.0000,662.4055,499.9000,inelastic
1,001829cb707d,1,-6.6005,2.7182,5.8068,145.9415,120.8179,elastic
2,001829cb707d,2,-2.0932,1.6667,2.3361,156.0676,126.1185,elastic
3,001829cb707d,4,-5.9090,1.7966,4.0381,149.2148,117.7095,elastic
4,001b302a9387,1,-8.8473,5.2143,5.7600,78.5087,77.5800,elastic


## Revenue Optimization Strategy

In [15]:
# Define optimal pricing strategy based on elasticity
def determine_pricing_strategy(elasticity_df):
    pricing_strategy = elasticity_df.copy()
    
    # For elastic items (>1 in absolute value) - small price decrease leads to larger quantity increase
    pricing_strategy['recommended_price_change'] = 0.0
    
    # Elastic items - can increase revenue by lowering price slightly
    elastic_mask = pricing_strategy['elasticity_category'] == 'elastic'
    pricing_strategy.loc[elastic_mask, 'recommended_price_change'] = -0.05  # 5% price reduction
    
    # Inelastic items - can increase revenue by raising price
    inelastic_mask = pricing_strategy['elasticity_category'] == 'inelastic'
    pricing_strategy.loc[inelastic_mask, 'recommended_price_change'] = 0.03  # 3% price increase
    
    # Abnormal items - need special handling
    abnormal_mask = pricing_strategy['elasticity_category'] == 'abnormal'
    pricing_strategy.loc[abnormal_mask, 'recommended_price_change'] = 0.0  # No change
    
    # Calculate recommended new price
    pricing_strategy['recommended_new_price'] = pricing_strategy['avg_price_normal'] * (1 + pricing_strategy['recommended_price_change'])
    
    # Calculate potential revenue impact
    pricing_strategy['current_revenue_per_item'] = pricing_strategy['avg_price_normal'] * pricing_strategy['avg_qty_normal']
    pricing_strategy['projected_revenue_per_item'] = pricing_strategy['recommended_new_price'] * (pricing_strategy['avg_qty_normal'] * (1 + pricing_strategy['elasticity'] * pricing_strategy['recommended_price_change']))
    pricing_strategy['revenue_impact_per_item'] = pricing_strategy['projected_revenue_per_item'] - pricing_strategy['current_revenue_per_item']
    
    return pricing_strategy

pricing_strategy_df = determine_pricing_strategy(elasticity_df)
print("Pricing strategy summary:")
print(pricing_strategy_df[['elasticity_category', 'recommended_price_change', 'revenue_impact_per_item']].groupby('elasticity_category').agg(['mean', 'count']))
pricing_strategy_df.head()

Pricing strategy summary:
                    recommended_price_change        revenue_impact_per_item  \
                                        mean  count                    mean   
elasticity_category                                                           
abnormal                              0.0000   8783                  0.0000   
elastic                              -0.0500  15402              98119.2569   
inelastic                             0.0300   5831                  9.2604   

                            
                     count  
elasticity_category         
abnormal              8783  
elastic              15402  
inelastic             5831  


,item_id,store_id,elasticity,avg_qty_normal,avg_qty_promo,avg_price_normal,avg_price_promo,elasticity_category,recommended_price_change,recommended_new_price,current_revenue_per_item,projected_revenue_per_item,revenue_impact_per_item
0,00179dda14f8,1,-0.0000,1.0000,1.0000,662.4055,499.9000,inelastic,0.0300,682.2777,662.4055,682.2777,19.8722
1,001829cb707d,1,-6.6005,2.7182,5.8068,145.9415,120.8179,elastic,-0.0500,138.6444,396.6955,501.2343,104.5388
2,001829cb707d,2,-2.0932,1.6667,2.3361,156.0676,126.1185,elastic,-0.0500,148.2642,260.1126,272.9692,12.8565
3,001829cb707d,4,-5.9090,1.7966,4.0381,149.2148,117.7095,elastic,-0.0500,141.7541,268.0808,329.9205,61.8397
4,001b302a9387,1,-8.8473,5.2143,5.7600,78.5087,77.5800,elastic,-0.0500,74.5833,409.3668,560.9338,151.5670


In [16]:
# Save pricing strategy for use in prediction model
pricing_strategy_df.to_csv('pricing_strategy.csv', index=False)
print("Pricing strategy saved to 'pricing_strategy.csv'")

Pricing strategy saved to 'pricing_strategy.csv'
